# 04 · Small SFT and RLVR loop on synthetic data

Trains Model A on synthetic credit excerpts, then compares it with the base model. The plan and
its limits are in `docs/next_steps.md`.

1. **Data:** 300 SFT, 200 RLVR and 60 held-out excerpts, built by code, so every answer is known.
2. **Short test:** a few real steps of each stage. It stops the notebook if the loss is broken,
   and cuts RLVR to 100 prompts if it would take over 4 hours.
3. **SFT:** QLoRA on Model A, 75 optimizer steps.
4. **Gate:** SFT must beat Model A's mean reward on the held-out set by at least 0.10, or RLVR is
   skipped.
5. **RLVR:** GRPO from the SFT adapter, 4 sampled answers per prompt.
6. **Comparison:** Model A, SFT and RLVR on the held-out set, and SFT and RLVR on the 5 M1
   agreements. Model A's M3 scores stay the real-agreement baseline.

**Kaggle:** Accelerator **GPU T4 x2** (not P100), Internet **on**, Add-ons → Secrets: attach
`HF_TOKEN` with write access. "Save & Run All" runs it in the background.
**Colab:** T4 runtime, add `HF_TOKEN` under 🔑 Secrets with notebook access enabled.

**Time (estimate, not measured):** 4 to 10 hours. The short test prints the real pace.

Resumable: adapters and decode records go to the private HF repos as they are made, and a restart
skips finished work. No Nebius call and no cost. All logic lives in `src/`.


In [ ]:
REPO_URL = "https://github.com/z26qin/credit-extractor.git"
BRANCH = "m4-small-loop"
INSTALL_VLLM = True      # vLLM goes into its own venv (/tmp/vllm-env); False -> transformers only


In [ ]:
import os, subprocess, sys
ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else (
    "/content" if os.path.exists("/content") else os.getcwd())
REPO = os.path.join(ROOT, "credit-extractor")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "-q", "--ff-only"], check=True)
os.chdir(REPO); sys.path.insert(0, REPO)
# register the repo on sys.path permanently, so `import src` survives a kernel restart
import site
with open(os.path.join(site.getsitepackages()[0], "credit_extractor.pth"), "w") as f:
    f.write(REPO + "\n")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

pip = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(pip + ["-r", "requirements.txt"], check=True)

# 1) platform Python (transformers backend): never install vLLM here. Its torch pin
#    breaks the preinstalled torchaudio, which transformers imports.
BASE_CHECK = ("import torch, transformers\n"
              "from transformers import Qwen3ForCausalLM\n"
              "print('base: torch', torch.__version__, 'cuda', torch.version.cuda,"
              " 'gpu', torch.cuda.is_available(), '| transformers', transformers.__version__)")
r = subprocess.run([sys.executable, "-c", BASE_CHECK], capture_output=True, text=True)
if r.returncode and "torchaudio" in r.stderr.lower():
    print("torchaudio does not match torch; removing it (not needed for text models)")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"])
    r = subprocess.run([sys.executable, "-c", BASE_CHECK], capture_output=True, text=True)
print(r.stdout or r.stderr[-1500:])
assert r.returncode == 0, "base env broken: Run -> Factory reset, then run all cells again"

# 2) vLLM in its own venv; uv picks the torch build that matches this GPU driver
VENV = "/tmp/vllm-env"; VPY = VENV + "/bin/python"
VLLM_CHECK = ("import torch, vllm\n"
              "print('vllm venv: vllm', vllm.__version__, '| torch', torch.__version__,"
              " 'cuda', torch.version.cuda, 'gpu', torch.cuda.is_available())")
if INSTALL_VLLM and not os.path.exists(VPY):
    subprocess.run(pip + ["uv"], check=True)
    uv = [sys.executable, "-m", "uv"]
    subprocess.run(uv + ["venv", "-q", VENV, "--python", sys.executable], check=True)
    r = subprocess.run(uv + ["pip", "install", "-q", "--python", VPY, "--torch-backend=auto",
                             "-r", "requirements-gpu.txt", "-r", "requirements.txt", "wrapt"],
                       capture_output=True, text=True)
    if r.returncode:
        print("vllm install FAILED -> transformers only\n" + r.stderr[-2000:])
        import shutil; shutil.rmtree(VENV, ignore_errors=True)
if os.path.exists(VPY):
    r = subprocess.run([VPY, "-c", VLLM_CHECK], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else
          "vllm venv not usable (runner will fall back to transformers):\n" + r.stderr[-1500:])
else:
    print("no vllm venv: runner uses transformers")

# 3) training in its own venv, with the versions the CPU tests ran on
TVENV = "/tmp/train-env"; TPY = TVENV + "/bin/python"
TRAIN_CHECK = ("import torch, transformers, peft, trl, bitsandbytes\n"
               "print('train venv: torch', torch.__version__, 'cuda', torch.version.cuda,"
               " 'gpu', torch.cuda.is_available(), '| transformers', transformers.__version__,"
               " '| peft', peft.__version__, '| trl', trl.__version__)")
if not os.path.exists(TPY):
    subprocess.run(pip + ["uv"], check=True)
    uv = [sys.executable, "-m", "uv"]
    subprocess.run(uv + ["venv", "-q", TVENV, "--python", "3.12"], check=True)
    r = subprocess.run(uv + ["pip", "install", "-q", "--python", TPY, "--torch-backend=auto",
                             "-r", "requirements-train.txt", "-r", "requirements.txt"],
                       capture_output=True, text=True)
    assert r.returncode == 0, "training install failed:\n" + r.stderr[-2000:]
r = subprocess.run([TPY, "-c", TRAIN_CHECK], capture_output=True, text=True)
print(r.stdout or r.stderr[-1500:])
assert r.returncode == 0, "training venv not usable"


In [ ]:
# after a kernel restart, start again from this cell (the cells below need `env`)
from src.platform import setup
env = setup()
assert env.data_repo, "HF_TOKEN missing: attach it as a secret (outputs must sync to the Hub)"
assert env.gpus, "no GPU visible: pick GPU T4 x2 (Kaggle) or a T4 runtime (Colab)"


In [ ]:
from pathlib import Path
from src import small_loop as loop
from src.train import load_cfg
cfg = load_cfg()
data = loop.build_data(env, cfg)      # same seed, same excerpts after a restart
print(data)


In [ ]:
# seconds per step, peak memory, and how many RLVR prompts to use
short = loop.short_test(env, cfg, data)


In [ ]:
sft = loop.train_sft(env, cfg, data)


In [ ]:
# the gate: decodes the held-out set with Model A and with SFT
held_out = loop.score_models(env, cfg, "held_out", {"Model A": cfg["base_model"], "SFT": sft}, data)
passed = loop.gate(held_out, cfg)


In [ ]:
rlvr = loop.train_rlvr(env, cfg, data, sft, short["rlvr_prompts"]) if passed else None


In [ ]:
# Model A's real-agreement row comes from the M3 review sheet, so it does not run again
real = None
if passed:
    held_out.update(loop.score_models(env, cfg, "held_out", {"RLVR": rlvr}, data))
    real = loop.score_models(env, cfg, "real", {"Model A": "m3", "SFT": sft, "RLVR": rlvr}, data)


In [ ]:
adapters = {"SFT": sft, **({"RLVR": rlvr} if passed else {})}
report = loop.write_report(env, cfg, data, held_out, real, short, adapters)
print(f"session wall time: {env.session_minutes():.1f} min")
